In [9]:
# import os

# os.environ["REQUESTS_CA_BUNDLE"] = "./certs/pwc-bundle.pem"
# os.environ["SSL_CERT_FILE"] = "./certs/pwc-bundle.pem"

In [10]:
import pandas as pd

pd.set_option("display.max_colwidth", None)   # show full cell content
pd.set_option("display.max_rows", None)       # optional, show all rows

In [11]:
import pandas as pd
import numpy as np

ID_COL = "TalentLink ID"

df = pd.read_excel("synthetic_staff_talentlink_examples_v5.xlsx", header=6)

# Clean blanks only (no row deletion except fully empty)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.dropna(how="all")

# Trim strings
obj_cols = df.select_dtypes(include=["object", "string"]).columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.astype("string").str.strip())

# ID as numeric (important for grouping)
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

# Parse job dates safely
for c in ["Job History Start Date", "Job History End Date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df.head(2)

/var/folders/8d/dffstnrn5js2s44pz0sfhd1w0000gn/T/ipykernel_60859/3984814146.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
/var/folders/8d/dffstnrn5js2s44pz0sfhd1w0000gn/T/ipykernel_60859/3984814146.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


,Local employee ID,TalentLink ID,Resource Name,Resource Status,Resource Type,Management level,Job level,Grade Code,Global LoS,Global Network,...,Area of Study,Academic Institution,In Process,Project Type Interest,Travel Interest,Additional Information,Profile Last Edited (YYYY/MM/DD),Unnamed_BC,Unnamed_BD,Unnamed_BE
0,886579303,983794,Allison Hill,Enabled,Employee,M3,J3,5,Practice 1,Network A,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024-04-22T00:00:00,r886579303,48932.125,|Synthetic profile filter in 'Enabled'| |Synthetic domain in 'Practice 1'|
1,886579303,983794,Allison Hill,Enabled,Employee,M3,J3,5,Practice 1,Network A,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2024-08-05T00:00:00,r886579303,48932.125,|Synthetic profile filter in 'Enabled'| |Synthetic domain in 'Practice 1'|


In [12]:
skills_per_id = df[df["Skill Level 1"].isin(["Skills"])]

skills_per_id_grouped = (
    skills_per_id.groupby("TalentLink ID", sort=False)["Capability Name"]
        .agg(list)
        .reset_index(name="skills")
)

skills_per_id_grouped.head()

,TalentLink ID,skills
0,983794,"[Data Governance, Machine Learning, Cloud Architecture, Cybersecurity Governance, Risk Assessment, Strategic Planning, Risk Assessment, Project Delivery, Data Governance, Business Analysis, Machine Learning, Internal Controls, SQL Optimization, Business Analysis, Data Governance, Cybersecurity Governance, Data Analysis, Process Improvement, Change Management, Regulatory Reporting, Change Management, Tax Compliance, Financial Modeling, Treasury Operations, Cloud Architecture, Data Governance, Project Delivery, SQL Optimization, Risk Assessment, Data Governance, Strategic Planning, Regulatory Reporting, Project Delivery, Audit Planning, SQL Optimization, Strategic Planning, Project Delivery, Process Improvement, Internal Controls, Strategic Planning, Data Analysis, Machine Learning, Financial Modeling, Tax Compliance, Cybersecurity Governance, Python Development, Cloud Architecture, Stakeholder Management, Cloud Architecture, Project Delivery, Data Governance, Project Delivery, Treasury Operations]"
1,1938483,"[Stakeholder Management, Stakeholder Management, Cybersecurity Governance, Data Analysis, Risk Assessment, Project Delivery, Process Improvement, Stakeholder Management, Project Delivery, Regulatory Reporting, Project Delivery, Data Governance, Process Improvement, Cloud Architecture, Data Analysis, Tax Compliance, Data Analysis, Cloud Architecture, Regulatory Reporting, Stakeholder Management, Financial Modeling, Machine Learning, Tax Compliance, Cybersecurity Governance, Internal Controls, Strategic Planning, Change Management]"
2,4842788,"[SQL Optimization, Cybersecurity Governance, Cybersecurity Governance, Tax Compliance, Internal Controls, Project Delivery, SQL Optimization, Audit Planning, Project Delivery, Internal Controls, Python Development, Machine Learning, Machine Learning, Audit Planning, Regulatory Reporting, Data Governance, Tax Compliance, Change Management, Tax Compliance, Regulatory Reporting, Tax Compliance]"
3,1538552,"[Python Development, Strategic Planning, Tax Compliance, Machine Learning, Stakeholder Management, Business Analysis, Regulatory Reporting, Audit Planning, SQL Optimization, Business Analysis, Cloud Architecture, Cybersecurity Governance, SQL Optimization, Internal Controls, Tax Compliance, Internal Controls, Python Development, Change Management, Data Governance, Internal Controls, Python Development, SQL Optimization, Audit Planning, Risk Assessment, Data Analysis, Process Improvement, SQL Optimization, Audit Planning, Cloud Architecture]"
4,52339391,"[Cybersecurity Governance, Change Management, Internal Controls, Strategic Planning, Python Development, Strategic Planning, Internal Controls, Cybersecurity Governance, Risk Assessment, Stakeholder Management, Treasury Operations, Business Analysis, Risk Assessment, Data Governance, Data Analysis, Treasury Operations, Cloud Architecture, Project Delivery, Project Delivery, Cloud Architecture, Data Analysis, Machine Learning, SQL Optimization, Stakeholder Management, Python Development, Internal Controls, Internal Controls, Python Development, Treasury Operations, Change Management, Regulatory Reporting, Data Analysis, SQL Optimization, Strategic Planning, SQL Optimization, Internal Controls, Business Analysis, Cloud Architecture, Cybersecurity Governance, Stakeholder Management, Project Delivery, Risk Assessment, Data Analysis, Project Delivery, Financial Modeling, Internal Controls, Treasury Operations, Regulatory Reporting, Strategic Planning, Stakeholder Management, Python Development, Treasury Operations, Treasury Operations, Audit Planning, Python Development, Risk Assessment, Python Development, Regulatory Reporting, Business Analysis, Audit Planning, Cloud Architecture, Stakeholder Management, Business Analysis, Data Governance, Regulatory Reporting, Regulatory Reporting, Internal Controls, Project Delivery, Audit Planning, Change Management, Internal Controls, Python Development, Treasury Operatio

In [13]:
#This works
language = df[df["Skill Level 1"].isin(["Language"])]

langauage_grouped = (
    language.groupby("TalentLink ID", sort=False)["Capability Name"]
    .agg(list)
    .reset_index(name="languages")
)

langauage_grouped.head()

,TalentLink ID,languages
0,983794,"[English, German]"
1,1938483,"[English, English, English, English, English]"
2,4842788,"[English, English, English, English, English]"
3,1538552,"[English, English, English]"
4,52339391,"[English, English]"


In [14]:
#This works
summary = df.drop_duplicates(subset=['TalentLink ID', 'Summary/bio'])

biography = (
    summary
        .groupby("TalentLink ID", sort=False)["Summary/bio"]
        .agg(list)
        .reset_index(name="Biography")
)

biography[biography["TalentLink ID"] == 97817]

,TalentLink ID,Biography


In [15]:
import pandas as pd
from itertools import zip_longest

# --- 0. columns we care about
cols = [
    "Job History Start Date",
    "Job History End Date",
    "Position",
    "Company",
    "Description"
]

# --- 1. select, dedupe, and copy
jobs_by_id = (
    df[[
        "TalentLink ID",
        *cols
    ]]
    .dropna(subset=["Description"])
    .drop_duplicates()
    .copy()
)

# --- 2. clean text columns safely
text_cols = ["Position", "Company", "Description"]
for c in text_cols:
    jobs_by_id.loc[:, c] = (
        jobs_by_id[c].astype(str)
        .str.replace(r"\n+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": None})
    )

# --- 3. parse & format dates keeping gaps as None
for c in ["Job History Start Date", "Job History End Date"]:
    jobs_by_id.loc[:, c] = pd.to_datetime(jobs_by_id[c], errors="coerce")
    formatted = jobs_by_id[c].dt.strftime("%Y-%m-%d")
    jobs_by_id.loc[:, c] = formatted.where(jobs_by_id[c].notna(), None)

# --- 4. optional sort so arrays are chronological
jobs_by_id = jobs_by_id.sort_values(
    ["TalentLink ID", "Job History Start Date", "Job History End Date"],
    na_position="last"
)

# --- 5. make parallel lists per ID
grouped = (
    jobs_by_id
      .groupby("TalentLink ID", sort=False)[cols]
      .agg(list)
      .reset_index()
)

# --- 6. convert parallel lists -> list-of-dicts (aligned; zip_longest protects against mismatched lengths)
def lists_to_jobs(row):
    lists = [row[c] for c in cols]
    jobs = []
    for start, end, pos, comp, desc in zip_longest(*lists, fillvalue=None):
        jobs.append({
            "Job History Start Date": start,
            "Job History End Date": end,
            "Position": pos,
            "Company": comp,
            "Description": desc
        })
    return jobs

grouped["Jobs"] = grouped.apply(lists_to_jobs, axis=1)

# --- 7. final frame: one row per ID with Jobs = list of aligned job dicts
projects = grouped[["TalentLink ID", "Jobs"]]

# explode for everyone while keeping TalentLink ID
person_jobs = projects.explode("Jobs").reset_index(drop=True)

# normalize the dict column (skip rows where Jobs is missing)
jobs_df = pd.concat(
    [
        person_jobs[["TalentLink ID"]],
        pd.json_normalize(person_jobs["Jobs"])
    ],
    axis=1
)


I think i should join up all the datasets now to a new big dataset.

In [16]:
master_dataset = (
    skills_per_id_grouped.set_index("TalentLink ID")
    .join(biography.set_index("TalentLink ID"))
    .join(jobs_df.set_index("TalentLink ID"))
)

In [17]:
master_dataset.to_csv("master_dataset.csv", index=True)

In [18]:
#This works
riskTeams = df[['TalentLink ID', 'Strategic Region']]
team_groups = riskTeams.drop_duplicates(subset=['TalentLink ID', 'Strategic Region'])


team = (
    team_groups
        .groupby("TalentLink ID", sort=False)["Strategic Region"]
        .agg(list)
        .reset_index(name="Team")
)

team.head(3)


,TalentLink ID,Team
0,983794,[Market 6]
1,1938483,[Market 2]
2,4842788,[Market 5]


Now i think i have all the data i really need to build the machine learning model. Its time to start small. I need to find a way to transform text into numbers. This means i need to look at the projects and biography vairables and see if i can extract more info

I think i will use the 4 steps i found in my preliminary research

In [19]:
master_dataframe = (
    master_dataset
    .groupby("TalentLink ID")
    .agg({
        "skills": "first",
        "Biography": "first",
        "Job History Start Date": list,
        "Job History End Date": list,
        "Position": list,
        "Company": list,
        "Description": list
    })
    .reset_index()
)

In [21]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

nlp = spacy.load("en_core_web_sm")


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

Stop word reomval

In [ ]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# load spaCy model
nlp = spacy.load("en_core_web_sm")

# =========================================================
# STEP 1: spaCy stop word removal
# =========================================================
def remove_stopwords(text):
    text = str(text)
    doc = nlp(text)
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

master_dataframe["step1_stopwords_removed"] = master_dataframe["Description"].apply(remove_stopwords)

# =========================================================
# STEP 2: spaCy POS tagging + lemmatization
# =========================================================
def tag_and_lemmatize(text):
    text = str(text)
    doc = nlp(text)

    lemmas = []
    pos_tags = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            lemmas.append(token.lemma_)
            pos_tags.append((token.text, token.pos_))

    return pd.Series([
        " ".join(lemmas),   # cleaned lemmatized text
        pos_tags            # token + POS tags
    ])

master_dataframe[["step2_lemmatized", "pos_tags"]] = master_dataframe["step1_stopwords_removed"].apply(tag_and_lemmatize)

# =========================================================
# STEP 3: TF-IDF vector representation
# =========================================================
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(master_dataframe["step2_lemmatized"])

# Optional: group by TalentLink ID
grouped_df = (
    master_dataframe
    .groupby("TalentLink ID")["step2_lemmatized"]
    .apply(" ".join)
    .reset_index()
)

X_grouped_tfidf = tfidf_vectorizer.fit_transform(grouped_df["step2_lemmatized"])

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# group employee text
employee_df = (
    master_dataframe
    .groupby("TalentLink ID", as_index=False)
    .agg({
        "step2_lemmatized": " ".join,
        "skills": "first"
    })
)

# create TF-IDF matrix
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform(employee_df["step2_lemmatized"])
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
tfidf_df["TalentLink ID"] = employee_df["TalentLink ID"]

# score only the employee's listed skills
skill_scores = []

for i, row in employee_df.iterrows():
    employee_id = row["TalentLink ID"]
    employee_skills = row["skills"]

    if not isinstance(employee_skills, list):
        employee_skills = []

    for skill in employee_skills:
        skill_lower = str(skill).lower()
        score = tfidf_df.loc[i, skill_lower] if skill_lower in tfidf_df.columns else 0

        skill_scores.append({
            "TalentLink ID": employee_id,
            "skill": skill,
            "score": score
        })

skill_scores_df = pd.DataFrame(skill_scores)

In [ ]:
# keep only skills that appear in descriptions
filtered = skill_scores_df[skill_scores_df["score"] > 0]

skill_analysis_table = (
    filtered
    .groupby("skill")
    .agg(
        employees_demonstrating=("TalentLink ID", "nunique"),
        average_score=("score", "mean"),
        max_score=("score", "max")
    )
    .sort_values("employees_demonstrating", ascending=False)
    .reset_index()
)

skill_analysis_table.head()

,skill,employees_demonstrating,average_score,max_score
0,Risk Management,16,0.048042,0.127513
1,Business Continuity,11,0.064255,0.160426
2,Communication,11,0.020496,0.047358
3,Technology,10,0.031819,0.115997
4,Insurance,8,0.029901,0.086396


What i need to do here is add biography/summary as part of the calculation 

In [ ]:
filtered.head()

,TalentLink ID,skill,score
44,103419,Investment Bank,0.084569
85,128641,Business Continuity,0.068386
92,128641,Communication,0.019065
97,128641,Design,0.117735
105,128641,Governance Risk,0.027000
